# Treinamento R(2+1)D + LSTM Multimodal — vídeo + áudio

Modelo usando **(2+1)D CNN** (`r2plus1d_18`, pré-treinada na
Kinetics-400) como extrator de vídeo. O áudio é processado pelo
AudioMAE (congelado).

### Diferença importante vs. a versão ResNet2D
A (2+1)D CNN processa o clipe inteiro com convoluções espaço-temporais, então
o eixo `T` é reduzido pelos strides temporais da rede (aprox. `T' = T // 8`
depois do stem + layer1..4). A sequência que chega na LSTM aqui é mais curta
do que na versão framewise, isso é esperado, já que a CNN 3D já captura
parte da dinâmica temporal sozinha; a LSTM modela uma temporalidade "mais
grossa" por cima.


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys, datetime, random, gc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

import optuna
from transformers import AutoModel
import einops
import timm
import torchaudio

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

from models.nn.r2plus1d_lstm_multimodal_modified import R2Plus1DLSTM_MultiModal
from preprocessing.loader_multimodal_frac2 import (
    build_multimodal_dataloader,
    train_video_transform,
    TARGET_SHAPE,
    train_mel_transform,
)
from preprocessing.balanced_dataset import balanced_df


/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
torch.cuda.empty_cache()

In [4]:
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    # Habilita TF32 para ops de matriz — sem perda de qualidade notável, +15 % velocidade
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # autotuner de kernels

SEED = 435
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Torch: 2.5.1+cu121
Torchvision: 0.20.1+cu121
CUDA disponível: True
CUDA: 12.1
GPU: NVIDIA RTX A4500
Device: cuda


## 2. Configuração

In [5]:
LABELS_ALL  = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_combinadas_wg.csv"
LABELS_DIR  = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used"
CKPT_DIR    = "/mnt/storage_C4/gaussian_football/models/checkpoints"
RUNS_DIR    = "/mnt/storage_C4/gaussian_football/runs_multimodal_r2plus1d_free_gpu"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# Treino
EPOCHS       = 100
PATIENCE     = 20
LR           = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
MONITOR      = "loss"   # "ccc" | "mae" | "loss"

# DataLoader 
BATCH_SIZE   = 1   # a (2+1)D CNN consome bem mais VRAM que a ResNet2D por causa
                    # das convoluções 3D — comece pequeno e ajuste GRAD_ACCUM/FRAME_STEP

# GPU / memória 
USE_AMP        = True   # Mixed Precision FP16 — principal economia de VRAM
USE_COMPILE    = True   # torch.compile (PyTorch >= 2.0); desative se der erro
                        # OBS: usar mode="reduce-overhead" (CUDA Graphs) junto com o
                        # freeze/unfreeze dinâmico do backbone e a troca train()/eval()
                        # do BatchNorm por época causava erro de operação in-place no
                        # autograd ("version X; expected version X-1") em quase todos os
                        # trials. Por isso o run_experiment abaixo usa mode="default".
GRAD_ACCUM     = 4      # acumula N mini-batches antes de step → simula batch*N
GRAD_CKPT      = True   # gradient checkpointing real nos blocos da (2+1)D CNN
FRAME_STEP     = 4      # subamostragem temporal ANTES da CNN 3D (1 = usa todos os frames)
FRAC_EPOCH     = 0.2    # usa 10 % dos grupos por época
NUM_WORKERS    = 2      # menos workers = menos vídeos simultâneos na RAM
MAX_CLIPS_PER_GROUP = 8 # teto de clipes por grupo (window_id); grupos maiores
                         # são subamostrados aleatoriamente. Sem isso, o "batch"
                         # real de um step é o tamanho do grupo (pode ter dezenas
                         # de clipes) — principal causa do CUDA OOM esporádico.
                         # Ajuste pra cima/baixo testando com nvidia-smi.

# Modelo
MODEL_DEFAULTS = dict(
    frame_step=FRAME_STEP,
    hidden_size=256,
    num_layers=1,
    use_dropout=True,
    dropout_p=0.3,
    in_channels=1, # grayscale
    use_grad_checkpoint=GRAD_CKPT,
    audio_chunk_size=16, # processa o AudioMAE em blocos de N espectrogramas
                          # em vez de B*T_mel de uma vez — reduz o pico de
                          # memória do forward do áudio quando o grupo é grande
)


## 3. Balanceando os Dados

In [6]:
FORCE_REBUILD = True

paths_balanced = {
    "train": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_train_wg.csv"),
    "valid": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_valid_wg.csv"),
    "test":  os.path.join(LABELS_DIR, "balanced_todas_as_ligas_test_wg.csv"),
}

if FORCE_REBUILD or not all(os.path.exists(p) for p in paths_balanced.values()):
    all_data = pd.read_csv(LABELS_ALL)
    for split, out_path in paths_balanced.items():
        subset  = all_data[all_data["split"] == split]
        balanced = balanced_df(subset, "game_id", threshold=0.5, random_state=SEED)
        balanced.to_csv(out_path, index=False)
        print(f"{split}: {len(balanced)} clipes -> {out_path}")
else:
    print("CSVs balanceados já existem.")


train: 16912 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_train_wg.csv
valid: 6152 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_valid_wg.csv
test: 5730 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_test_wg.csv


## 4. DataLoaders Multimodais

In [7]:
dir_background_goal = '/mnt/storage_C4/gaussian_football/data/datasets_goal_background'

# atualiza os paths que o dataloader vai ler
# CORRIGIDO: usa os CSVs balanceados gerados na Seção 3 (paths_balanced),
# em vez da variável indefinida 'dir_background_goal' (resíduo de outro notebook)
TRAIN_PATH = paths_balanced["train"]
VAL_PATH   = paths_balanced["valid"]
TEST_PATH  = paths_balanced["test"]

print("\nTRAIN_PATH, VAL_PATH, TEST_PATH atualizados:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)


TRAIN_PATH, VAL_PATH, TEST_PATH atualizados:
  /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_train_wg.csv
  /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_valid_wg.csv
  /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_test_wg.csv


In [8]:
GROUPS            = True
GROUPS_COLUMN_ID  = "window_id"

def _make_loader(csv_path, split, pair, batch_size, shuffle, epoch_frac=1.0):
    """Fábrica de DataLoader com configurações GPU-otimizadas.

    epoch_frac default = 1.0 (usa todos os grupos). CORRIGIDO: antes o
    val_loader também recebia epoch_frac=FRAC_EPOCH (0.1) por causa do
    default do parâmetro do _make_loader antigo, que amarrava treino e
    validação na mesma fração. Como GroupSampler/PairGroupSampler sorteiam
    um subconjunto novo a cada __iter__ (chamado 1x por época mesmo com
    shuffle=False), a validação estava rodando em 10% dos grupos, sorteados
    de novo a cada época — nunca no mesmo conjunto duas vezes. Isso mascarava
    qualquer tendência real de aprendizado atrás do ruído de amostragem de um
    val set minúsculo e mutável, e fazia o ReduceLROnPlateau/early stopping
    reagirem a esse ruído em vez de a uma tendência real.
    """
    return build_multimodal_dataloader(
        csv_path=csv_path,
        pair=pair,
        split=split,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        is_grayscale=True,
        pin_memory=False,       # True esgota RAM fixada com vídeos grandes
        groups=GROUPS,
        column_groups_id=GROUPS_COLUMN_ID,
        video_transform=train_video_transform if shuffle else None,
        mel_transform=train_mel_transform if shuffle else None,
        target_shape=TARGET_SHAPE,
        epoch_frac=epoch_frac,
        frame_step=FRAME_STEP,
        max_clips_per_group=MAX_CLIPS_PER_GROUP,
    )

train_loader = _make_loader(TRAIN_PATH, "train", pair=True,  batch_size=BATCH_SIZE, shuffle=True,
                             epoch_frac=FRAC_EPOCH)
val_loader   = _make_loader(VAL_PATH,   "valid", pair=True,  batch_size=BATCH_SIZE, shuffle=False,
                             epoch_frac=1.0)


Dataset de Treino: 16178/16912 exemplos válidos.
4456 grupos encontrados.
Low: 8089
High: 8089

26 pares de grupos criados.

Dataset de Validação: 5362/6152 exemplos válidos.
1393 grupos encontrados.
Low: 2681
High: 2681

10 pares de grupos criados.



## 5. Métricas

In [9]:
def calculate_ccc(y_true, y_pred):
    """Concordance Correlation Coefficient sobre dois arrays 1-D."""
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true,  var_pred  = np.var(y_true),  np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    denom = var_true + var_pred + (mean_true - mean_pred) ** 2
    return 0.0 if denom == 0 else (2 * cov) / denom

def binary_accuracy(y_true, y_pred, threshold=0.5):
    return float(np.mean((y_pred > threshold) == (y_true > threshold)))

def pairwise_rank_accuracy(low_pred, high_pred):
    """Fração dos pares em que high_pred > low_pred.

    Métrica mais diagnóstica que binary_accuracy pra esse setup: mede
    exatamente o que a margin ranking loss está otimizando (a acc binária a
    0.5 é calculada sobre pares artificialmente balanceados pela construção
    do dataset pareado, então não reflete a distribuição real de
    arousal_score nem corresponde diretamente ao que a loss combinada
    otimiza). Tende a mostrar sinal de aprendizado mesmo quando a calibração
    absoluta (BCE) ainda está ruim.
    """
    return float(np.mean(np.array(high_pred) > np.array(low_pred)))


## 6. Loss Combinada (BCE + Margin Ranking Adaptativa)

In [10]:
class CombinedLoss(nn.Module):
    """BCE + Margin Ranking Loss com margem adaptativa."""

    def __init__(self, alpha=1.0, margin_scale=1.0):
        super().__init__()
        self.alpha = alpha
        self.margin_scale = margin_scale
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, low_label, high_label, low_pred, high_pred, return_components=False):
        bce = (self.bce(low_pred, low_label) + self.bce(high_pred, high_label)) / 2
        margin = self.margin_scale * (high_label - low_label)
        rank = torch.relu(margin - (high_pred - low_pred)).mean()
        loss = bce + self.alpha * rank
        return (loss, bce, rank) if return_components else loss


## 7. Treino GPU-Otimizado

In [11]:
MONITOR_MODE = {"ccc": "max", "mae": "min", "loss": "min"}


def _apply_freeze(model, freeze_backbone):
    """Liga/desliga gradiente da (2+1)D CNN; o stem (índice 0 de model.cnn,
    que inclui o primeiro conv adaptado para grayscale) sempre treina."""
    for name, param in model.cnn.named_parameters():
        param.requires_grad = name.startswith("0.") or (not freeze_backbone)


def _set_backbone_eval(model):
    """Congela o BatchNorm em TODA a (2+1)D CNN.
    Isso evita o erro in-place do num_batches_tracked durante o gradient checkpointing."""
    for module in model.cnn.modules():
        # Avalia apenas se é uma camada de BatchNorm e trava no modo eval
        if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
            module.eval()


def _is_better(current, best, mode):
    return current > best if mode == "max" else current < best


def _log_pred_scatter(writer, y_true, y_pred, tag, step):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(y_true, y_pred, s=8, alpha=0.4)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "--", color="gray", linewidth=1)
    ax.set_xlabel("target"); ax.set_ylabel("pred"); ax.set_title("pred vs target")
    writer.add_figure(tag, fig, step)
    plt.close(fig)


def train_model(
    model, train_loader, val_loader, *,
    criterion, run_name, optimizer, scheduler,
    loss_name,
    freeze_mode="finetune", unfreeze_epoch=5,
    freeze_bn_always=True,
    epochs=EPOCHS, patience=PATIENCE, monitor=MONITOR,
    grad_clip=GRAD_CLIP,
    use_amp=USE_AMP,
    accum_steps=GRAD_ACCUM,
    checkpoint_path=None, device=device,
    trial=None,
    lr=LR,
):
    """
    Treina e valida por época com Mixed Precision + acumulação de gradiente.
    """
    model.to(device)
    gc.collect()
    torch.cuda.empty_cache()

    if checkpoint_path is None:
        checkpoint_path = os.path.join(CKPT_DIR, f"{run_name}.pth")
    last_path = os.path.join(CKPT_DIR, f"{run_name}_last.pth")

    mode = MONITOR_MODE[monitor]
    best_score = -np.inf if mode == "max" else np.inf
    best_metrics, best_true, best_pred = {}, None, None
    epochs_no_improve = 0

    # GradScaler só é relevante com AMP
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    stamp   = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(RUNS_DIR, f"{run_name}_{stamp}")
    writer  = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):
        # Freeze schedule
        if freeze_mode == "full":
            backbone_frozen = False
        elif freeze_mode == "frozen":
            backbone_frozen = True
        else:  # finetune
            backbone_frozen = epoch < unfreeze_epoch
            if epoch == unfreeze_epoch:
                print(f"[época {epoch+1}] descongelando backbone (fine-tuning completo)")

        _apply_freeze(model, backbone_frozen)

        # Treino
        model.train()
        if freeze_bn_always or backbone_frozen:
            _set_backbone_eval(model)

        train_loss = train_bce = train_rank = 0.0
        train_true_buf, train_pred_buf = [], []
        optimizer.zero_grad(set_to_none=True)  # set_to_none economiza memória

        for step, (low, high) in enumerate(
            tqdm(train_loader, desc=f"época {epoch+1}/{epochs} [treino]", leave=False)
        ):
            low_video,  _, low_mel,  low_targets  = low
            high_video, _, high_mel, high_targets = high

            # Transferência assíncrona (non_blocking=True): CPU não espera GPU
            low_video   = low_video.to(device,  non_blocking=True)
            high_video  = high_video.to(device, non_blocking=True)
            low_mel     = low_mel.to(device,    non_blocking=True)
            high_mel    = high_mel.to(device,   non_blocking=True)
            low_targets = low_targets.to(device).float().view(-1)
            high_targets= high_targets.to(device).float().view(-1)

            with torch.amp.autocast("cuda", enabled=use_amp):
                low_out  = model(low_video,  low_mel).reshape(-1)
                high_out = model(high_video, high_mel).reshape(-1)
                loss, bce_l, rank_l = criterion(
                    low_targets, high_targets, low_out, high_out,
                    return_components=True,
                )
                # Normaliza pela acumulação para gradientes comparáveis
                loss = loss / accum_steps

            scaler.scale(loss).backward()

            if (step + 1) % accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    grad_clip,
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            bs = low_video.size(0)
            # Detach + cpu imediato libera VRAM do tensor de saída
            preds   = torch.cat([torch.sigmoid(low_out), torch.sigmoid(high_out)]).detach().cpu()
            targets = torch.cat([low_targets, high_targets]).detach().cpu()

            train_loss += loss.item() * accum_steps * bs
            train_bce  += bce_l.item() * bs
            train_rank += rank_l.item() * bs

            train_true_buf.extend(targets.numpy())
            train_pred_buf.extend(preds.numpy())

        # Passo residual se o dataset não for múltiplo de accum_steps
        if (len(train_loader) % accum_steps) != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), grad_clip
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        n = len(train_loader.dataset)
        train_loss /= n; train_bce /= n; train_rank /= n

        train_true = np.array(train_true_buf)
        train_pred = np.array(train_pred_buf)
        train_mae  = np.mean(np.abs(train_true - train_pred))
        train_ccc  = calculate_ccc(train_true, train_pred)
        train_acc  = binary_accuracy(train_true, train_pred)

        # Validação 
        model.eval()
        val_loss = val_bce = val_rank = 0.0
        all_true, all_pred = [], []
        # buffers separados por lado do par, para a acurácia de ranking
        # (fração de pares em que high_pred > low_pred) — a métrica que
        # reflete diretamente o que a margin ranking loss otimiza
        val_low_pred_buf, val_high_pred_buf = [], []

        with torch.no_grad():
            for (low, high) in tqdm(val_loader, desc=f"época {epoch+1}/{epochs} [val]", leave=False):
                low_video,  _, low_mel,  low_targets  = low
                high_video, _, high_mel, high_targets = high

                low_video   = low_video.to(device,  non_blocking=True)
                high_video  = high_video.to(device, non_blocking=True)
                low_mel     = low_mel.to(device,    non_blocking=True)
                high_mel    = high_mel.to(device,   non_blocking=True)
                low_targets = low_targets.to(device).float().view(-1)
                high_targets= high_targets.to(device).float().view(-1)

                with torch.amp.autocast("cuda", enabled=use_amp):
                    low_out  = model(low_video,  low_mel).reshape(-1)
                    high_out = model(high_video, high_mel).reshape(-1)
                    loss, bce_l, rank_l = criterion(
                        low_targets, high_targets, low_out, high_out,
                        return_components=True,
                    )

                bs = low_video.size(0)
                val_loss += loss.item() * bs
                val_bce  += bce_l.item() * bs
                val_rank += rank_l.item() * bs

                low_prob  = torch.sigmoid(low_out).cpu()
                high_prob = torch.sigmoid(high_out).cpu()
                preds   = torch.cat([low_prob, high_prob])
                targets = torch.cat([low_targets, high_targets]).cpu()
                all_true.extend(targets.numpy())
                all_pred.extend(preds.numpy())
                val_low_pred_buf.extend(low_prob.numpy())
                val_high_pred_buf.extend(high_prob.numpy())

        n = len(val_loader.dataset)
        val_loss /= n; val_bce /= n; val_rank /= n

        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        val_mae  = np.mean(np.abs(all_true - all_pred))
        val_ccc  = calculate_ccc(all_true, all_pred)
        val_acc  = binary_accuracy(all_true, all_pred)
        val_rank_acc = pairwise_rank_accuracy(val_low_pred_buf, val_high_pred_buf)

        scheduler.step(val_loss)

        # TensorBoard 
        writer.add_scalars("loss",      {"train": train_loss, "val": val_loss},  epoch)
        writer.add_scalars("bce",       {"train": train_bce,  "val": val_bce},   epoch)
        writer.add_scalars("rank_loss", {"train": train_rank, "val": val_rank},  epoch)
        writer.add_scalars("ccc",       {"train": train_ccc,  "val": val_ccc},   epoch)
        writer.add_scalars("mae",       {"train": train_mae,  "val": val_mae},   epoch)
        writer.add_scalars("acc",       {"train": train_acc,  "val": val_acc},   epoch)
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)

        print(
            f"[{epoch+1:03d}] loss={val_loss:.4f} bce={val_bce:.4f} rank={val_rank:.4f} "
            f"ccc={val_ccc:.4f} mae={val_mae:.4f} rank_acc={val_rank_acc:.3f}"
        )

        # Checkpoint / early-stopping
        current_score = {"ccc": val_ccc, "mae": val_mae, "loss": val_loss}[monitor]
        torch.save(model.state_dict(), last_path)

        if _is_better(current_score, best_score, mode):
            best_score   = current_score
            best_metrics = {
                "val_ccc": val_ccc, "val_mae": val_mae,
                "val_loss": val_loss, "val_acc": val_acc,
                "val_bce": val_bce, "val_rank": val_rank,
                "val_rank_acc": val_rank_acc,
            }
            best_true, best_pred = all_true, all_pred
            torch.save(model.state_dict(), checkpoint_path)
            epochs_no_improve = 0
            print(f"  ✓ novo melhor {monitor}: {best_score:.4f} → salvo em {checkpoint_path}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping na época {epoch+1}")
                break

        # Optuna pruning
        if trial is not None:
            trial.report(current_score, epoch)
            if trial.should_prune():
                writer.close()
                raise optuna.TrialPruned()

        # Libera tensores intermediários que possam ter ficado no grafo
        gc.collect()

    if best_true is not None:
        _log_pred_scatter(writer, best_true, best_pred, "best_pred_scatter", 0)

    writer.add_hparams(
        {
            "backbone": "r2plus1d_18", "lr": lr,
            "loss": loss_name, "monitor": monitor,
        },
        {
            "best/val_loss": best_metrics.get("val_loss", np.inf),
            "best/val_ccc":  best_metrics.get("val_ccc",  0.0),
            "best/val_acc":  best_metrics.get("val_acc",  0.0),
        },
        run_name=".",
    )
    writer.close()
    torch.cuda.empty_cache()
    print(f"Concluído. Melhor {monitor}: {best_score:.4f}")
    return {"checkpoint": checkpoint_path, "best_metrics": best_metrics}


## 8. Função `run_experiment`

In [12]:
def free_gpu_memory():
    """Libera o máximo possível da VRAM usada até aqui.

    Além de gc.collect()+torch.cuda.empty_cache(), também reseta o cache do
    torch.compile (torch._dynamo.reset()) — sem isso, os guards/artefatos
    compilados de um trial continuam vivos e retendo VRAM quando o próximo
    trial compila um modelo novo, então a memória "vaza" de trial pra trial
    em vez de ser liberada. Chamada depois de CADA trial do Optuna (sucesso,
    prune ou erro).
    """
    if USE_COMPILE and hasattr(torch, "_dynamo"):
        try:
            torch._dynamo.reset()
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def run_experiment(
    audiomae,
    alpha=1.0,
    margin_scale=1.0,
    freeze_mode="finetune",
    unfreeze_epoch=5,
    lr=LR,
    lr_backbone=None,
    weight_decay=WEIGHT_DECAY,
    model_kwargs=None,
    criterion=None,
    epochs=EPOCHS,
    patience=PATIENCE,
    use_amp=USE_AMP,
    accum_steps=GRAD_ACCUM,
    use_fusion=True,
    always_bn=False,
    train_loader=None,
    val_loader=None,
    trial=None,
):
    model_kwargs = {**MODEL_DEFAULTS, **(model_kwargs or {})}
    run_name = (
        f"r2plus1d18__{freeze_mode}__fusion{use_fusion}"
        f"__bn{always_bn}__trial{trial.number if trial else 'manual'}"
    )
    print(f"\n=== {run_name} ===")

    model = R2Plus1DLSTM_MultiModal(
        audiomae,
        use_fusion=use_fusion,
        **model_kwargs,
    ).to(device)

    # torch.compile: funde ops e elimina overhead Python no forward
    if USE_COMPILE and hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="default")
        except Exception as e:
            print(f"[AVISO] torch.compile falhou ({e}); usando modelo sem compilação.")

    if lr_backbone is None:
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = AdamW(
            [
                {"params": model.cnn.parameters(),  "lr": lr_backbone},
                {"params": model.lstm.parameters(), "lr": lr},
                {"params": model.head.parameters(), "lr": lr},
            ],
            weight_decay=weight_decay,
        )

    scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

    if criterion is None:
        criterion = CombinedLoss(alpha=alpha, margin_scale=margin_scale)

    try:
        result = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            run_name=run_name,
            optimizer=optimizer,
            scheduler=scheduler,
            loss_name=criterion.__class__.__name__,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=epochs,
            patience=patience,
            use_amp=use_amp,
            accum_steps=accum_steps,
            freeze_bn_always=always_bn,
            lr=lr,
            trial=trial,
        )
    finally:
        # Garantia de limpeza mesmo em caso de exceção
        del model, optimizer, scheduler, criterion
        free_gpu_memory()

    return result


## 9. AudioMAE — carrega e congela uma vez

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_ae = AutoModel.from_pretrained("hance-ai/audiomae", trust_remote_code=True).to(device)

# Congela o AudioMAE permanentemente (não requer grad em nenhum trial)
model_ae.eval()
for p in model_ae.parameters():
    p.requires_grad = False

# Confirma memória disponível antes de começar os trials
if torch.cuda.is_available():
    free_mb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_allocated()) / 1024**2
    print(f"VRAM livre antes dos trials: {free_mb:.0f} MB")


VRAM livre antes dos trials: 19851 MB


## 10. Teste rápido (sanity check) antes da busca

Roda um único batch pelo modelo para conferir shapes e detectar erros antes
de gastar tempo na busca de hiperparâmetros — útil principalmente aqui porque
a (2+1)D CNN tem requisitos de shape diferentes da ResNet2D (clipe inteiro
em vez de frame a frame).

In [14]:
_sanity_model = R2Plus1DLSTM_MultiModal(model_ae, **MODEL_DEFAULTS).to(device)
_sanity_model.eval()

with torch.no_grad():
    (low, high) = next(iter(train_loader))
    low_video, _, low_mel, low_targets = low
    low_video = low_video.to(device)
    low_mel = low_mel.to(device)
    print("video:", low_video.shape, "mel:", low_mel.shape)
    print(f"tamanho do grupo (deve ser <= MAX_CLIPS_PER_GROUP={MAX_CLIPS_PER_GROUP}):", low_video.shape[0])

    # pra n crashar
    with torch.amp.autocast("cuda", enabled=USE_AMP):
        out = _sanity_model(low_video, low_mel)

    print("saída do modelo:", out.shape)
    # diagnóstico: T_sub é o comprimento da sequência que chega na BiLSTM
    # após o pooling temporal da (2+1)D CNN. Se ficar em 1-2, a LSTM está
    # processando sequências praticamente sem dimensão temporal — não tem o
    # que aprender ali, e vale revisar FRAME_STEP / duração dos clipes antes
    # de investigar outra coisa.
    print(f"T_sub (comprimento da sequência antes da BiLSTM): {_sanity_model.last_T_sub}")

del _sanity_model
gc.collect()
torch.cuda.empty_cache()


video: torch.Size([5, 23, 1, 224, 398]) mel: torch.Size([5, 1, 128, 256])
tamanho do grupo (deve ser <= MAX_CLIPS_PER_GROUP=8): 5
saída do modelo: torch.Size([5, 1])
T_sub (comprimento da sequência antes da BiLSTM): 3


## 11. Busca Optuna com Pruning Hyperband

In [ ]:
SEARCH_EPOCHS = 40


def objective(trial):
    freeze_mode = trial.suggest_categorical("freeze_mode", ["frozen", "finetune"])
    use_fusion  = trial.suggest_categorical("use_fusion",  [True, False])
    always_bn   = trial.suggest_categorical("always_bn",   [True, False])
    lr          = trial.suggest_float("lr",           1e-5, 1e-3, log=True)
    alpha       = trial.suggest_float("alpha",        0.0, 1.0)
    margin_scale= trial.suggest_float("margin_scale", 0.5, 2.0)

    # try/finally em volta do trial inteiro: free_gpu_memory() roda sempre,
    # independente do trial terminar com sucesso, ser podado (TrialPruned)
    # ou lançar exceção — garantindo que a VRAM usada por este trial não
    # "vaze" pro próximo.
    try:
        result = run_experiment(
            audiomae=model_ae,
            freeze_mode=freeze_mode,
            unfreeze_epoch=5,
            alpha=alpha,
            margin_scale=margin_scale,
            use_fusion=use_fusion,
            always_bn=always_bn,
            lr=lr,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=SEARCH_EPOCHS,
            trial=trial,
        )
        return result["best_metrics"]["val_rank"]
    except optuna.TrialPruned:
        raise
    except Exception as e:
        print(f"ERRO no trial {trial.number}: {e}")
        return float("inf")
    finally:
        # run_experiment já libera model/optimizer/scheduler/criterion no
        # seu próprio finally; chamamos de novo aqui como garantia extra e
        # pra reportar a VRAM livre após o trial.
        free_gpu_memory()
        if torch.cuda.is_available():
            free_mb = (torch.cuda.get_device_properties(0).total_memory
                       - torch.cuda.memory_allocated()) / 1024**2
            print(f"[trial {trial.number}] VRAM livre após limpeza: {free_mb:.0f} MB")


study = optuna.create_study(
    direction="minimize",
    study_name="multimodal_r2plus1d",
    storage="sqlite:///optuna_multimodal_r2plus1d.db",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.HyperbandPruner(min_resource=5, max_resource=SEARCH_EPOCHS),
)

study.optimize(objective, n_trials=20)

print("\n=== MELHOR ===")
print(study.best_params)
print(f"val_rank: {study.best_value:.4f}")


[I 2026-07-02 11:08:52,485] Using an existing study with name 'multimodal_r2plus1d' instead of creating a new one.



=== r2plus1d18__frozen__fusionFalse__bnTrue__trial235 ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal_r2plus1d_free_gpu/r2plus1d18__frozen__fusionFalse__bnTrue__trial235_20260702-110853


[001] loss=6.8290 bce=4.0094 rank=5.4166 ccc=0.0036 mae=0.4257 rank_acc=0.603
  ✓ novo melhor loss: 6.8290 → salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/r2plus1d18__frozen__fusionFalse__bnTrue__trial235.pth


[002] loss=6.8071 bce=3.9994 rank=5.3936 ccc=0.0055 mae=0.4252 rank_acc=0.655
  ✓ novo melhor loss: 6.8071 → salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/r2plus1d18__frozen__fusionFalse__bnTrue__trial235.pth


[003] loss=6.7904 bce=3.9908 rank=5.3781 ccc=0.0070 mae=0.4249 rank_acc=0.672
  ✓ novo melhor loss: 6.7904 → salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/r2plus1d18__frozen__fusionFalse__bnTrue__trial235.pth


...

[001] loss=5.7236 bce=3.9873 rank=5.6811 ccc=0.0035 mae=0.4283 rank_acc=0.586
  ✓ novo melhor loss: 5.7236 → salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/r2plus1d18__finetune__fusionTrue__bnFalse__trial236.pth


[002] loss=5.6853 bce=3.9625 rank=5.6370 ccc=0.0045 mae=0.4248 rank_acc=0.603
  ✓ novo melhor loss: 5.6853 → salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/r2plus1d18__finetune__fusionTrue__bnFalse__trial236.pth


[003] loss=5.6608 bce=3.9461 rank=5.6106 ccc=0.0079 mae=0.4242 rank_acc=0.603
  ✓ novo melhor loss: 5.6608 → salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/r2plus1d18__finetune__fusionTrue__bnFalse__trial236.pth


[004] loss=5.6169 bce=3.9291 rank=5.5225 ccc=0.0152 mae=0.4224 rank_acc=0.707
  ✓ novo melhor loss: 5.6169 → salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/r2plus1d18__finetune__fusionTrue__bnFalse__trial236.pth


[005] loss=5.5863 bce=3.9172 rank=5.4614 ccc=0.0204 mae=0.4212 rank_acc=0.741


## 12. Relatório dos Resultados

In [ ]:
import optuna.visualization as vis

# Top-5 trials
df_trials = study.trials_dataframe()
print(df_trials.sort_values("value").head(5).to_string(index=False))

# Importância dos hiperparâmetros
try:
    importances = optuna.importance.get_param_importances(study)
    for param, imp in importances.items():
        print(f"  {param:<20s}: {imp:.3f}")
except Exception as e:
    print(f"Importâncias indisponíveis: {e}")


 number    value             datetime_start          datetime_complete               duration  params_alpha  params_always_bn params_freeze_mode  params_lr  params_margin_scale  params_use_fusion  system_attrs_completed_rung_0  system_attrs_completed_rung_1    state
    116 0.423070 2026-07-01 18:28:31.991588 2026-07-01 18:42:14.118083 0 days 00:13:42.126495      0.314866              True             frozen   0.000054             0.971232               True                       0.476805                            NaN COMPLETE
    115 0.568871 2026-07-01 18:22:44.758499 2026-07-01 18:28:31.946274 0 days 00:05:47.187775      0.267918              True             frozen   0.000043             1.914838               True                       0.571116                       0.568871   PRUNED
    117 0.584384 2026-07-01 18:42:14.181780 2026-07-01 18:49:40.098487 0 days 00:07:25.916707      0.332404              True             frozen   0.000036             1.353555               True    